# TrackNetV2-padel — Test de detección de pelota

Prueba la arquitectura TrackNetV2 (Keras/TF) con los pesos originales de bádminton
sobre vídeo de pádel. Genera un CSV con detecciones y vídeo anotado.

**Nota:** Los pesos son los originales de bádminton (no hay pesos padel públicos).
El objetivo es ver si la arquitectura detecta algo útil antes de hacer fine-tuning.

**Requisitos:** GPU (T4 recomendado, también funciona en CPU aunque lento)

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'Sin GPU — correrá en CPU (lento)')

In [ ]:
# Instalar dependencias
!pip install gdown opencv-python-headless pillow scikit-learn imutils pandas --quiet
# TF2 ya viene en Colab, verificar
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Clonar repo
import os
if not os.path.exists('TrackNetV2-padel'):
    !git clone https://github.com/SamuReyes/TrackNetV2-padel.git --quiet
    print('Clonado.')
else:
    print('Ya existe.')

In [ ]:
# Descargar pesos originales de TrackNetV2 (bádminton)
# Fuente primaria: servidor NYCU. Fuente alternativa: mirrors conocidos.
import gdown, os

os.makedirs('weights', exist_ok=True)

# Los pesos del repo original de TrackNetV2 (bádminton, 3_in_1_out)
# Hay varios mirrors disponibles — probamos en orden
weight_path = 'weights/model_33'

if not os.path.exists(weight_path):
    print('Intentando descargar pesos desde Google Drive (mirror)...')
    # Mirror del modelo 3_in_1_out de TrackNetV2 (bádminton original)
    try:
        gdown.download(
            'https://drive.google.com/uc?id=1oKuGCd4AkE7LmXBUiqJoWfFUxb0pJjmH',
            output=weight_path,
            quiet=False
        )
        print('Descargado desde mirror.')
    except Exception as e:
        print(f'Mirror falló: {e}')
        print('Intentando servidor NYCU...')
        !wget -q "https://nol.cs.nctu.edu.tw:234/open-source/TrackNetv2/blob/master/3_in_1_out/model_33" -O weights/model_33 || echo 'NYCU no accesible'
else:
    print('Pesos ya descargados.')

# Verificar descarga
if os.path.exists(weight_path) and os.path.getsize(weight_path) > 1000:
    print(f'✅ Pesos OK ({os.path.getsize(weight_path)/1e6:.1f} MB)')
else:
    print('❌ Pesos no disponibles — sube el archivo model_33 manualmente en la siguiente celda')

In [ ]:
# Si la descarga automática falló, sube el archivo model_33 manualmente
# (descárgalo de https://github.com/SamuReyes/TrackNetV2-padel o del repo original)
# Comenta esta celda si ya tienes el archivo

import os
if not os.path.exists('weights/model_33') or os.path.getsize('weights/model_33') < 1000:
    from google.colab import files
    print('Sube el archivo model_33:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, 'weights/model_33')
    print('Archivo subido.')
else:
    print('model_33 ya disponible, saltando subida.')

In [ ]:
# Subir vídeo de prueba
from google.colab import files
print('Sube tu vídeo de prueba (ej: test_60s.mp4):')
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f'Vídeo: {VIDEO_PATH}')

In [ ]:
# Cargar modelo TrackNetV2 (arquitectura Keras)
import sys
sys.path.insert(0, 'TrackNetV2-padel/3_in_1_out')

import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
import keras.backend as K
from keras.models import load_model

def custom_loss(y_true, y_pred):
    loss = (-1) * (
        K.square(1 - y_pred) * y_true * K.log(K.clip(y_pred, K.epsilon(), 1)) +
        K.square(y_pred) * (1 - y_true) * K.log(K.clip(1 - y_pred, K.epsilon(), 1))
    )
    return K.mean(loss)

print('Cargando modelo...')
try:
    model = load_model('weights/model_33', custom_objects={'custom_loss': custom_loss})
    print('✅ Modelo cargado correctamente')
    model.summary()
except Exception as e:
    print(f'❌ Error cargando modelo: {e}')
    print('Verifica que el archivo model_33 es válido y no está corrupto')

In [ ]:
# Inferencia sobre el vídeo
from tqdm import tqdm

HEIGHT, WIDTH = 288, 512
MAX_FRAMES = 500  # Ajusta para test más largo
OUTPUT_VIDEO = '/tmp/tracknet_padel_result.mp4'
OUTPUT_CSV   = '/tmp/tracknet_padel_result.csv'

cap = cv2.VideoCapture(VIDEO_PATH)
vid_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
total  = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), MAX_FRAMES)
ratio  = vid_h / HEIGHT
print(f'Vídeo: {vid_w}x{vid_h} @ {fps:.0f}fps — procesando {total} frames')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (vid_w, vid_h))

csv_lines = ['Frame,Visibility,X,Y']
detected = 0
processed = 0

def read_and_resize(cap):
    ok, frame = cap.read()
    if not ok:
        return None, None
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (WIDTH, HEIGHT))
    return frame, resized

# Leer los 3 primeros frames
f1_orig, f1 = read_and_resize(cap)
f2_orig, f2 = read_and_resize(cap)
f3_orig, f3 = read_and_resize(cap)

if f1 is None or f2 is None or f3 is None:
    print('Error leyendo vídeo')
else:
    out.write(f1_orig)
    out.write(f2_orig)
    frame_idx = 2

    pbar = tqdm(total=total)
    pbar.update(3)

    while f3 is not None and (MAX_FRAMES == 0 or frame_idx < MAX_FRAMES):
        # Preparar input: (1, 9, H, W) — 3 frames × 3 canales
        def to_channels_first(img):
            return np.moveaxis(img.astype('float32') / 255.0, -1, 0)  # (3, H, W)

        x1 = to_channels_first(f1)
        x2 = to_channels_first(f2)
        x3 = to_channels_first(f3)

        unit = np.concatenate([x1, x2, x3], axis=0)  # (9, H, W)
        unit = unit.reshape(1, 9, HEIGHT, WIDTH)

        y_pred = model.predict(unit, batch_size=1, verbose=0)
        y_pred = (y_pred > 0.5).astype('float32')
        heatmap = (y_pred[0] * 255).astype('uint8')

        processed += 1
        annotated = f3_orig.copy()

        if heatmap.max() > 0:
            # Encontrar centro de la detección
            cnts, _ = cv2.findContours(heatmap.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            rects = [cv2.boundingRect(c) for c in cnts]
            if rects:
                best = max(rects, key=lambda r: r[2] * r[3])
                cx = int(ratio * (best[0] + best[2] / 2))
                cy = int(ratio * (best[1] + best[3] / 2))
                detected += 1
                csv_lines.append(f'{frame_idx},1,{cx},{cy}')
                cv2.circle(annotated, (cx, cy), 12, (0, 255, 0), 3)
                cv2.putText(annotated, 'BALL', (cx + 15, cy),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
        else:
            csv_lines.append(f'{frame_idx},0,0,0')

        det_rate = detected / processed * 100
        cv2.putText(annotated, f'Det: {det_rate:.1f}% ({detected}/{processed})',
                    (12, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (200, 200, 200), 3)
        out.write(annotated)

        # Avanzar ventana deslizante
        f1, f1_orig = f2, f2_orig
        f2, f2_orig = f3, f3_orig
        f3_orig, f3 = read_and_resize(cap)
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    out.release()

    # Guardar CSV
    with open(OUTPUT_CSV, 'w') as f:
        f.write('\n'.join(csv_lines))

    print(f'\n=== RESULTADO ===')
    print(f'Frames procesados: {processed}')
    print(f'Frames con pelota detectada: {detected}')
    print(f'Detection rate: {detected/processed*100:.1f}%  (referencia YOLOv8: ~20%)')
    print(f'Vídeo guardado: {OUTPUT_VIDEO}')
    print(f'CSV guardado: {OUTPUT_CSV}')

In [ ]:
# Análisis del CSV — ¿dónde está detectando la pelota?
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(OUTPUT_CSV)
visible = df[df['Visibility'] == 1]

print(f'Frames totales: {len(df)}')
print(f'Frames detectados: {len(visible)} ({len(visible)/len(df)*100:.1f}%)')

if len(visible) > 0:
    print(f'\nDistribución de posición X: min={visible.X.min():.0f} max={visible.X.max():.0f} media={visible.X.mean():.0f}')
    print(f'Distribución de posición Y: min={visible.Y.min():.0f} max={visible.Y.max():.0f} media={visible.Y.mean():.0f}')

    # Plot de trayectoria detectada
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.scatter(visible['X'], visible['Y'], alpha=0.5, s=10, c='lime')
    ax1.set_xlim(0, vid_w)
    ax1.set_ylim(vid_h, 0)  # Invertir Y (origen arriba-izquierda)
    ax1.set_facecolor('#1a1a2e')
    ax1.set_title('Posiciones detectadas en pista', color='white')
    ax1.tick_params(colors='white')
    fig.patch.set_facecolor('#1a1a2e')

    ax2.plot(visible['Frame'], visible['X'], alpha=0.7, label='X', color='cyan')
    ax2.plot(visible['Frame'], visible['Y'], alpha=0.7, label='Y', color='lime')
    ax2.set_title('Trayectoria X/Y por frame', color='white')
    ax2.legend()
    ax2.tick_params(colors='white')

    plt.tight_layout()
    plt.savefig('/tmp/trajectory.png', dpi=100, facecolor='#1a1a2e')
    plt.show()
    print('\nSi la trayectoria parece aleatoria → falsos positivos')
    print('Si la trayectoria parece física (parábolas, rebotes) → detecciones reales')
else:
    print('\n⚠️ Sin detecciones — el modelo no supera el threshold de 0.5 en ningún frame')
    print('Prueba a bajar el threshold en el código de inferencia (y_pred > 0.3)')

In [ ]:
# Descargar resultados
from google.colab import files
print('Descargando vídeo anotado...')
files.download(OUTPUT_VIDEO)
print('Descargando CSV de detecciones...')
files.download(OUTPUT_CSV)
print('Listo.')